In [ ]:
# Resolve defaults from this project's root, independently of the working directory.
from pathlib import Path as _KSPath
import sys as _ks_sys
from functools import partial as _ks_partial
_ks_start = _KSPath.cwd()
_ks_root = next((p for p in (_ks_start, *_ks_start.parents) if (p / 'kinspec_paths.py').is_file()), None)
if _ks_root is None:
    raise RuntimeError('Cannot locate the KinSPEC project root containing kinspec_paths.py. Start the notebook kernel from the project directory or a subdirectory.')
_ks_sys.path.insert(0, str(_ks_root))
from kinspec_paths import ROOT as KINSPEC_ROOT, project_path as _ks_project_path
_kp = _ks_partial(_ks_project_path, script_dir='feature_embedding/mol')
import os as _ks_os
_ks_os.chdir(KINSPEC_ROOT / 'feature_embedding/mol')



In [ ]:
# ===== Imports =====
from __future__ import annotations
import os, gc, json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from tqdm import tqdm

# -------------------------------------------------
# 1. Configuration
# -------------------------------------------------
@dataclass(frozen=True)
class SAEConfig:
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_dim: int       = 4096
    latent_dim: int      = 1024
    # csv_path: str        =  "../../dataset/KinSPEC_training_pool_merged_4_datasets_deduplicated_with_smiles.csv"
    csv_path: str        =  _kp('dataset/KinSPEC_training_pool_morgan_one_hot.csv')
    chunk_rows: int      = 50_000
    batch_size: int      = 1024
    model_path: str      = _kp('feature_embedding/mol/LSAE_chembl_1900k_full_dataset_v4_model.pt')
    idx_path: str        = _kp('feature_embedding/mol/LSAE_chembl_1900k_full_dataset_v4_topk_of1024d_top99.pt')
    # output
    out_decoder_pt: str  = _kp('feature_embedding/mol/LSAE_training_pool_of1024d_decoder_top99_v4.pt')
    output_csv: str      = _kp('dataset/KinSPEC_training_pool_LSAE_mol_latent_top95_features_v4.csv')
# -------------------------------------------------
# 2. LSAE
# -------------------------------------------------
class SparseLinearAE(nn.Module):
    def __init__(self, cfg: SAEConfig):
        super().__init__()
        self.encoder = nn.Linear(cfg.input_dim, cfg.latent_dim, bias=True)
        self.decoder = nn.Linear(cfg.latent_dim, cfg.input_dim, bias=True)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

# -------------------------------------------------
# 3. Load model & latent index
# -------------------------------------------------
def load_model_and_idx(cfg: SAEConfig):
    print(f"[info] Using device: {cfg.device}")
    
    model = SparseLinearAE(cfg).to(cfg.device)
    checkpoint = torch.load(cfg.model_path, map_location=cfg.device, weights_only=True)
    model.load_state_dict(checkpoint)
    model.eval()

    keep_idx = torch.load(cfg.idx_path, map_location=cfg.device, weights_only=True).long()

    #----------------
    keep_idx = keep_idx[:417]#extract top-k latent of 95% bit=1 coverage 417
    #------------------

    print(f"[info] loaded model + {len(keep_idx)} latent indices on {cfg.device}")
    return model, keep_idx

# -------------------------------------------------
# 4. deduplicate and mapping
# -------------------------------------------------
def deduplicate_csv(csv_path: str, chunk_size: int = 50000) -> Tuple[pd.DataFrame, np.ndarray]:

    print("[info] Reading and deduplicating CSV...")
    
    # read chunk and deduplicate
    unique_rows = {}  # hash -> (row_data, first_occurrence_idx)
    row_to_unique_idx = []  
    unique_count = 0
    total_rows = 0
    
    with open(csv_path, 'rb') as f:
        total_lines = sum(1 for _ in f)
    
    # chunk
    for chunk in tqdm(pd.read_csv(csv_path, chunksize=chunk_size,header=None), 
                      desc="Deduplicating", total=(total_lines//chunk_size + 1)):
        
        for idx, row in chunk.iterrows():
            row_hash = hashlib.md5(row.values.tobytes()).hexdigest()
            
            if row_hash not in unique_rows:
                unique_rows[row_hash] = (row.values, unique_count)
                row_to_unique_idx.append(unique_count)
                unique_count += 1
            else:
                _, existing_unique_idx = unique_rows[row_hash]
                row_to_unique_idx.append(existing_unique_idx)
            
            total_rows += 1
    
    # create deduplicated DataFrame
    unique_data = np.array([data for data, _ in unique_rows.values()])
    unique_df = pd.DataFrame(unique_data)
    
    print(f"[info] Original rows: {total_rows}, Unique rows: {unique_count}")
    print(f"[info] Deduplication ratio: {unique_count/total_rows:.3f}")
    
    return unique_df, np.array(row_to_unique_idx)

# -------------------------------------------------
# 5. simplified dataset
# -------------------------------------------------
class UniqueFingerprintDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.data = torch.from_numpy(df.values.astype(np.uint8)).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# -------------------------------------------------
# 6. optimised extract function
# -------------------------------------------------
def extract_and_save_with_dedup(cfg: SAEConfig):
    unique_df, mapping = deduplicate_csv(cfg.csv_path, cfg.chunk_rows)
    model, keep_idx = load_model_and_idx(cfg)
    unique_ds = UniqueFingerprintDataset(unique_df)
    loader = DataLoader(
        unique_ds, 
        batch_size=cfg.batch_size, 
        num_workers=0,
        pin_memory=True if cfg.device.type == 'cuda' else False
    )
    
    unique_feats: List[torch.Tensor] = []
    
    print(f"[info] Extracting features from {len(unique_ds)} unique molecules...")
    with torch.no_grad():
        for xb in tqdm(loader, desc=f"encoding unique molecules on {cfg.device}"):
            xb = xb.to(cfg.device, non_blocking=True)
            z = model.encoder(xb)[:, keep_idx]  # [B, k]
            unique_feats.append(z.cpu())
            
            # clean GPU memory
            if len(unique_feats) % 20 == 0:
                torch.cuda.empty_cache() if cfg.device.type == 'cuda' else None
    
    # 4. concat unified feature
    unique_latent = torch.cat(unique_feats).numpy()  # [unique_count, k]
    print(f"[info] Unique features shape: {unique_latent.shape}")
    
    # 5. restore the original order based on the mapping
    print("[info] Restoring original order...")
    original_latent = unique_latent[mapping]  # [original_count, k]
    print(f"[info] Restored features shape: {original_latent.shape}")
    
    # 6. save

    pd.DataFrame(original_latent).to_csv(cfg.output_csv, index=False)
    print(f"[save] latent features → {cfg.output_csv}")
    
    # # save dedupicated data (optional)
    # dedup_info = {
    #     "unique_count": len(unique_df),
    #     "original_count": len(mapping),
    #     "mapping": mapping,
    #     "dedup_ratio": len(unique_df) / len(mapping)
    # }
    
    # dedup_info_path = "E:\\Study\\Code\\DDM_workplace\\data_source\\SPCA\\SAE_model\\mol_chembl\\dedup_info.json"
    # with open(dedup_info_path, 'w') as f:
    #     json.dump({k: v.tolist() if isinstance(v, np.ndarray) else v 
    #               for k, v in dedup_info.items()}, f)
    # print(f"[save] deduplication info → {dedup_info_path}")
    
    # save decoder
    dec_sub = model.decoder.weight[:, keep_idx].cpu()
    decoder_data = {
        "idx": keep_idx.cpu(), 
        "decoder_sub": dec_sub, 
        "bias": model.decoder.bias.cpu()
    }
    torch.save(decoder_data, cfg.out_decoder_pt)
    print(f"[save] decoder submatrix → {cfg.out_decoder_pt}")
    
    # clean gpu memory
    if cfg.device.type == 'cuda':
        torch.cuda.empty_cache()
        print(f"[info] GPU memory cleared")

# -------------------------------------------------
# 7. main
# -------------------------------------------------
if __name__ == "__main__":
    cfg = SAEConfig()
    
    if cfg.device.type == 'cuda':
        print(f"[info] GPU: {torch.cuda.get_device_name()}")
        print(f"[info] GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    extract_and_save_with_dedup(cfg)

[info] GPU: NVIDIA GeForce RTX 5090
[info] GPU Memory: 33.7 GB
[info] Reading and deduplicating CSV...


Deduplicating: 100%|██████████| 14/14 [01:56<00:00,  8.35s/it]


[info] Original rows: 692704, Unique rows: 300534
[info] Deduplication ratio: 0.434
[info] Using device: cuda
[info] loaded model + 417 latent indices on cuda
[info] Extracting features from 300534 unique molecules...


encoding unique molecules on cuda: 100%|██████████| 294/294 [00:00<00:00, 297.62it/s]


[info] Unique features shape: (300534, 417)
[info] Restoring original order...
[info] Restored features shape: (692704, 417)
[save] latent features → ../../dataset/KinSPEC_training_pool_LSAE_mol_latent_top95_features_v4.csv
[save] decoder submatrix → ./LSAE_training_pool_of1024d_decoder_top99_v4.pt
[info] GPU memory cleared


: 